# 🏆 DeepSeek-R1-Distill-Qwen-7B: Full GSM8K Benchmark Suite
This notebook runs an end-to-end, apples-to-apples benchmark on the complete **GSM8K test set (1,319 samples)** across three variants:
1. **Baseline Model** (`deepseek-ai/DeepSeek-R1-Distill-Qwen-7B`)
2. **SFT Model** (`hari31416/deepseek-r1-7b-grug-adapters`, subfolder `sft`)
3. **DPO Model** (`hari31416/deepseek-r1-7b-grug-adapters`, subfolder `dpo`)

All evaluations feature:
- Single-GPU pin (`device_map={"": 0}`) to avoid multi-GPU mismatch errors
- Defensive, real-time incremental saving after each model phase
- Publication-quality comparative dashboard plot generation
- Comprehensive JSON and Markdown export


In [ ]:
import os
import sys

# 1. Prevent multi-GPU tensor splitting errors by pinning exclusively to GPU 0
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. Install essential dependencies quietly
!pip install -q "peft>=0.14.0" "transformers>=4.48.0" "bitsandbytes>=0.45.0" "accelerate>=1.2.0" "datasets>=3.0.0" matplotlib seaborn pandas

print("CUDA Device Count:", 1 if os.environ.get("CUDA_VISIBLE_DEVICES") else "all")


In [ ]:
import gc
import re
import time
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Set random seed for reproducibility
torch.manual_seed(42)

# Output directory configuration
OUTPUT_DIR = "/kaggle/working"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

STYLE_SYSTEM_PROMPT = (
    "You are a helpful assistant. You must think in short, telegraphic, "
    "bullet-point style fragments inside a <think>...</think> block before answering."
)

def extract_numeric_answer(text: str):
    """Extract numerical answer from GSM8K format #### <number> or text fallback."""
    if not text:
        return None
    match = re.search(r"####\s*(-?[\$]?[0-9,]+(?:\.[0-9]+)?)", text)
    if match:
        raw = match.group(1).replace("$", "").replace(",", "").strip()
        try:
            return float(raw)
        except ValueError:
            pass
    # Fallback to the last number in the string
    numbers = re.findall(r"-?[0-9]+(?:\.[0-9]+)?", text)
    if numbers:
        try:
            return float(numbers[-1])
        except ValueError:
            pass
    return None

def parse_thinking_and_answer(text: str):
    """Extract contents of <think>...</think> and following answer."""
    think_match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
    if think_match:
        thinking = think_match.group(1).strip()
        answer = text[think_match.end():].strip()
        is_compliant = True
    else:
        thinking = ""
        answer = text.strip()
        is_compliant = False
    return thinking, answer, is_compliant

print("Helper functions initialized.")


In [ ]:
print("Loading GSM8K test split...")
dataset = load_dataset("openai/gsm8k", "main", split="test")
print(f"Loaded {len(dataset)} evaluation samples.")


In [ ]:
def evaluate_model(
    model,
    tokenizer,
    dataset,
    model_name: str,
    batch_size: int = 8,
    limit: int = None,
    system_prompt: str = STYLE_SYSTEM_PROMPT,
):
    """Run batched evaluation over the dataset with defensive saving."""
    print(f"\n=======================================================")
    print(f"🚀 Starting Benchmark for: {model_name}")
    print(f"=======================================================")

    samples = dataset if limit is None else dataset.select(range(min(limit, len(dataset))))
    total_samples = len(samples)

    results = []
    correct_count = 0
    format_compliant_count = 0
    thinking_tokens_list = []
    answer_tokens_list = []
    total_tokens_list = []
    latencies = []

    start_time_all = time.time()

    for idx in range(0, total_samples, batch_size):
        batch = samples[idx : idx + batch_size]
        prompts = []
        ground_truths = []

        for q, a in zip(batch["question"], batch["answer"]):
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": q.strip()},
            ]
            formatted_prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            prompts.append(formatted_prompt)
            ground_truths.append(extract_numeric_answer(a))

        # Tokenize batch
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1536)
        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}

        t0 = time.time()
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.6,
                top_p=0.95,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        batch_latency = time.time() - t0
        per_sample_latency = batch_latency / len(prompts)

        # Process each item in batch
        for b_idx in range(len(prompts)):
            input_len = inputs["input_ids"][b_idx].shape[0]
            gen_tokens = output_ids[b_idx][input_len:]
            raw_response = tokenizer.decode(gen_tokens, skip_special_tokens=True)

            thinking, answer, compliant = parse_thinking_and_answer(raw_response)
            pred_num = extract_numeric_answer(answer)
            gold_num = ground_truths[b_idx]

            is_correct = (pred_num is not None and gold_num is not None and abs(pred_num - gold_num) < 1e-4)

            n_think = len(tokenizer.encode(thinking, add_special_tokens=False))
            n_ans = len(tokenizer.encode(answer, add_special_tokens=False))
            n_total = len(gen_tokens)

            if is_correct:
                correct_count += 1
            if compliant:
                format_compliant_count += 1

            thinking_tokens_list.append(n_think)
            answer_tokens_list.append(n_ans)
            total_tokens_list.append(n_total)
            latencies.append(per_sample_latency)

            results.append({
                "index": idx + b_idx,
                "question": batch["question"][b_idx],
                "ground_truth_raw": batch["answer"][b_idx],
                "ground_truth_numeric": gold_num,
                "prediction_numeric": pred_num,
                "is_correct": bool(is_correct),
                "is_format_compliant": bool(compliant),
                "thinking_tokens": n_think,
                "answer_tokens": n_ans,
                "total_tokens": n_total,
                "latency_sec": per_sample_latency,
            })

        processed = len(results)
        if processed % 40 == 0 or processed == total_samples:
            acc = (correct_count / processed) * 100
            fc = (format_compliant_count / processed) * 100
            print(f"⏳ [{model_name}] Evaluated {processed}/{total_samples} | Acc: {acc:.1f}% | Format: {fc:.1f}%")

    total_time = time.time() - start_time_all
    summary = {
        "model_name": model_name,
        "sample_count": total_samples,
        "correct_count": correct_count,
        "accuracy": correct_count / total_samples,
        "format_compliant_count": format_compliant_count,
        "format_compliance_rate": format_compliant_count / total_samples,
        "mean_thinking_tokens": float(np.mean(thinking_tokens_list)),
        "mean_answer_tokens": float(np.mean(answer_tokens_list)),
        "mean_total_tokens": float(np.mean(total_tokens_list)),
        "mean_latency": float(np.mean(latencies)),
        "total_evaluation_time_sec": total_time,
    }

    # Defensive saving
    save_payload = {"summary": summary, "results": results}
    json_path_results = os.path.join(RESULTS_DIR, f"{model_name}_gsm8k.json")
    json_path_root = os.path.join(OUTPUT_DIR, f"{model_name}_gsm8k.json")
    for p in [json_path_results, json_path_root]:
        with open(p, "w", encoding="utf-8") as f:
            json.dump(save_payload, f, indent=2)

    print(f"✅ [{model_name}] Completed! Accuracy: {summary[accuracy]*100:.2f}%. Saved to {json_path_root}")
    return summary


In [ ]:
BASE_MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
HF_ADAPTER_REPO = "hari31416/deepseek-r1-7b-grug-adapters"

print(f"Loading Base Model: {BASE_MODEL_ID} in 4-bit (pinned to GPU 0)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)
print("Base model loaded successfully onto GPU 0.")


In [ ]:
# 1. Evaluate Baseline Model (Base 7B without adapters)
base_summary = evaluate_model(
    model=base_model,
    tokenizer=tokenizer,
    dataset=dataset,
    model_name="baseline",
    batch_size=8,
)

gc.collect()
torch.cuda.empty_cache()


In [ ]:
# 2. Attach SFT Adapter and Evaluate
print(f"Attaching SFT Adapter from {HF_ADAPTER_REPO} (subfolder=sft)...")
sft_model = PeftModel.from_pretrained(
    base_model,
    HF_ADAPTER_REPO,
    subfolder="sft",
    is_trainable=False,
)

sft_summary = evaluate_model(
    model=sft_model,
    tokenizer=tokenizer,
    dataset=dataset,
    model_name="sft",
    batch_size=8,
)

# Unload SFT adapter from memory
sft_model = sft_model.unload()
del sft_model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# 3. Attach DPO Adapter and Evaluate
print(f"Attaching DPO Adapter from {HF_ADAPTER_REPO} (subfolder=dpo)...")
dpo_model = PeftModel.from_pretrained(
    base_model,
    HF_ADAPTER_REPO,
    subfolder="dpo",
    is_trainable=False,
)

dpo_summary = evaluate_model(
    model=dpo_model,
    tokenizer=tokenizer,
    dataset=dataset,
    model_name="dpo",
    batch_size=8,
)

# Unload DPO adapter
dpo_model = dpo_model.unload()
del dpo_model
del base_model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Compile unified benchmark metrics
comparison_data = [base_summary, sft_summary, dpo_summary]
summary_df = pd.DataFrame(comparison_data)

summary_json_path = os.path.join(OUTPUT_DIR, "gsm8k_all_models_summary.json")
with open(summary_json_path, "w") as f:
    json.dump(comparison_data, f, indent=2)

print("\n" + "="*70)
print("📊 UNIFIED GSM8K BENCHMARK RESULTS (1,319 SAMPLES)")
print("="*70)
print(summary_df[["model_name", "accuracy", "format_compliance_rate", "mean_thinking_tokens", "mean_answer_tokens", "mean_latency"]])

# Generate 4-panel publication dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.set_theme(style="whitegrid")

models = ["Baseline", "SFT", "DPO"]
palette = ["#4A90E2", "#50E3C2", "#F5A623"]

# 1. Accuracy
accs = [base_summary["accuracy"] * 100, sft_summary["accuracy"] * 100, dpo_summary["accuracy"] * 100]
bars1 = axes[0, 0].bar(models, accs, color=palette, width=0.5)
axes[0, 0].set_title("GSM8K Accuracy (%)", fontsize=14, fontweight="bold")
axes[0, 0].set_ylabel("Accuracy (%)")
axes[0, 0].set_ylim(0, 100)
for bar in bars1:
    yval = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{yval:.1f}%", ha="center", va="bottom", fontweight="bold")

# 2. Thinking Tokens
thinks = [base_summary["mean_thinking_tokens"], sft_summary["mean_thinking_tokens"], dpo_summary["mean_thinking_tokens"]]
bars2 = axes[0, 1].bar(models, thinks, color=palette, width=0.5)
axes[0, 1].set_title("Mean Thinking Tokens (Reasoning Length)", fontsize=14, fontweight="bold")
axes[0, 1].set_ylabel("Tokens")
for bar in bars2:
    yval = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2.0, yval + 3, f"{yval:.1f}", ha="center", va="bottom", fontweight="bold")

# 3. Answer Tokens
answers = [base_summary["mean_answer_tokens"], sft_summary["mean_answer_tokens"], dpo_summary["mean_answer_tokens"]]
bars3 = axes[1, 0].bar(models, answers, color=palette, width=0.5)
axes[1, 0].set_title("Mean Answer Tokens (Brevity)", fontsize=14, fontweight="bold")
axes[1, 0].set_ylabel("Tokens")
for bar in bars3:
    yval = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2.0, yval + 2, f"{yval:.1f}", ha="center", va="bottom", fontweight="bold")

# 4. Latency
lats = [base_summary["mean_latency"], sft_summary["mean_latency"], dpo_summary["mean_latency"]]
bars4 = axes[1, 1].bar(models, lats, color=palette, width=0.5)
axes[1, 1].set_title("Mean Latency per Sample (seconds)", fontsize=14, fontweight="bold")
axes[1, 1].set_ylabel("Seconds")
for bar in bars4:
    yval = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, f"{yval:.2f}s", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "benchmark_comparison_dashboard.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"✅ Saved comparison dashboard to: {plot_path}")


In [ ]:
# Generate Markdown Report
report_md = f"""# DeepSeek-R1-Distill-Qwen-7B GSM8K Benchmark Report

Evaluated on the full GSM8K test split ({base_summary[sample_count]} samples).

| Model | Accuracy | Format Compliance | Mean Thinking Tokens | Mean Answer Tokens | Mean Total Tokens | Mean Latency |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Baseline** | {base_summary[accuracy]*100:.2f}% | {base_summary[format_compliance_rate]*100:.1f}% | {base_summary[mean_thinking_tokens]:.1f} | {base_summary[mean_answer_tokens]:.1f} | {base_summary[mean_total_tokens]:.1f} | {base_summary[mean_latency]:.2f}s |
| **SFT** | {sft_summary[accuracy]*100:.2f}% | {sft_summary[format_compliance_rate]*100:.1f}% | {sft_summary[mean_thinking_tokens]:.1f} | {sft_summary[mean_answer_tokens]:.1f} | {sft_summary[mean_total_tokens]:.1f} | {sft_summary[mean_latency]:.2f}s |
| **DPO** | {dpo_summary[accuracy]*100:.2f}% | {dpo_summary[format_compliance_rate]*100:.1f}% | {dpo_summary[mean_thinking_tokens]:.1f} | {dpo_summary[mean_answer_tokens]:.1f} | {dpo_summary[mean_total_tokens]:.1f} | {dpo_summary[mean_latency]:.2f}s |

## Key Findings
- **Reasoning Compression**: Measured difference in thinking tokens between Baseline and fine-tuned models.
- **Answer Brevity**: Measured reduction in conversational filler in the final response.
- **Accuracy Preservation**: Evaluated task accuracy retention on math problem-solving.
"""

report_path = os.path.join(OUTPUT_DIR, "BENCHMARK_REPORT.md")
with open(report_path, "w") as f:
    f.write(report_md)
print(f"✅ Generated Markdown report at: {report_path}")

# Package into ZIP archive
import zipfile
zip_path = os.path.join(OUTPUT_DIR, "kaggle_benchmark_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for target in ["results", "baseline_gsm8k.json", "sft_gsm8k.json", "dpo_gsm8k.json", "gsm8k_all_models_summary.json", "benchmark_comparison_dashboard.png", "BENCHMARK_REPORT.md"]:
        full_target = os.path.join(OUTPUT_DIR, target)
        if os.path.exists(full_target):
            if os.path.isdir(full_target):
                for root, _, files in os.walk(full_target):
                    for file in files:
                        fp = os.path.join(root, file)
                        zipf.write(fp, os.path.relpath(fp, OUTPUT_DIR))
            else:
                zipf.write(full_target, target)

print(f"📦 Successfully created downloadable archive: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)")
print("🎉 Benchmark pipeline execution completed successfully!")
